# Experimentos

Este proyecto compara tres algoritmos de aprendizaje automático (dos de clasificación y uno de regresión)
usando un diseño experimental 3×3×3 sobre los hiperparámetros principales.
El objetivo es evaluar rendimiento sin alto costo computacional.

**Algoritmos:**
- LinearRegression (regresión)
- LogisticRegression (clasificación)
- DecisionTreeClassifier (clasificación)

**Estructura de datos:**
- Train / Validation / Test


In [14]:
# ==========================================
# 1. Importación de librerías y carga de datos
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# Cargar datos ya preparados desde el notebook Prep.ipynb
df = pd.read_csv("../Data/video_game_reviews.csv")

# Vista general
df.head()


,Game Title,User Rating,Age Group Targeted,Price,Platform,Requires Special Device,Developer,Publisher,Release Year,Genre,Multiplayer,Game Length (Hours),Graphics Quality,Soundtrack Quality,Story Quality,User Review Text,Game Mode,Min Number of Players
0,Grand Theft Auto V,36.4,All Ages,41.41,PC,No,Game Freak,Innersloth,2015,Adventure,No,55.3,Medium,Average,Poor,"Solid game, but too many bugs.",Offline,1
1,The Sims 4,38.3,Adults,57.56,PC,No,Nintendo,Electronic Arts,2015,Shooter,Yes,34.6,Low,Poor,Poor,"Solid game, but too many bugs.",Offline,3
2,Minecraft,26.8,Teens,44.93,PC,Yes,Bungie,Capcom,2012,Adventure,Yes,13.9,Low,Good,Average,"Great game, but the graphics could be better.",Offline,5
3,Bioshock Infinite,38.4,All Ages,48.29,Mobile,Yes,Game Freak,Nintendo,2015,Sports,No,41.9,Medium,Good,Excellent,"Solid game, but the graphics could be better.",Online,4
4,Half-Life: Alyx,30.1,Adults,55.49,PlayStation,Yes,Game Freak,Epic Games,2022,RPG,Yes,13.2,High,Poor,Good,"Great game, but too many bugs.",Offline,1


In [15]:
# ==========================================
# 2. Definición de variables
# ==========================================

# Variable dependiente (objetivo)
target = 'User Rating'

# Variables independientes (todas menos la de salida)
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target]

# División del dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Variables independientes:", features[:5], "...")
print("Variable dependiente:", target)


Variables independientes: ['Game Title', 'Age Group Targeted', 'Price', 'Platform', 'Requires Special Device'] ...
Variable dependiente: User Rating


In [16]:
# ==========================================
# 2.5 Codificación de variables categóricas
# ==========================================

# Detectar columnas no numéricas
cat_cols = X.select_dtypes(include=['object', 'category']).columns
print("Columnas categóricas:", list(cat_cols))

# One-Hot Encoding automático (ligero)
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Actualizar división en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print(f"Nuevo tamaño del dataset codificado: {X_encoded.shape}")


Columnas categóricas: ['Game Title', 'Age Group Targeted', 'Platform', 'Requires Special Device', 'Developer', 'Publisher', 'Genre', 'Multiplayer', 'Graphics Quality', 'Soundtrack Quality', 'Story Quality', 'User Review Text', 'Game Mode']
Nuevo tamaño del dataset codificado: (47774, 99)


In [17]:
# ==========================================
# 3. Escalamiento de datos
# ==========================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [18]:
# ==========================================
# 4. Definición de modelos y sus hiperparámetros
# ==========================================

modelos = {
    "Regresión Lineal": {
        "modelo": LinearRegression(),
        "param_grid": {
            "fit_intercept": [True, False],
            "positive": [False, True]
        }
    },
    "Random Forest": {
        "modelo": RandomForestRegressor(random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "max_depth": [5, 10, 15],
            "min_samples_split": [2, 5, 10]
        }
    },
    "XGBoost": {
        "modelo": XGBRegressor(objective='reg:squarederror', random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.05, 0.1, 0.2],
            "max_depth": [3, 6, 9]
        }
    }
}


In [19]:
# ==========================================
# 5. Entrenamiento y evaluación con GridSearchCV
# ==========================================

resultados = []

for nombre, info in modelos.items():
    print(f"Entrenando modelo: {nombre}")
    
    grid = GridSearchCV(info["modelo"], info["param_grid"],
                        scoring='neg_mean_absolute_error', cv=3, n_jobs=-1)
    grid.fit(X_train_scaled, y_train)
    
    # Mejor modelo
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    resultados.append({
        "Modelo": nombre,
        "Mejores parámetros": grid.best_params_,
        "MAE": mae,
        "RMSE": rmse
    })

print("Entrenamiento finalizado.")


Entrenando modelo: Regresión Lineal


Entrenando modelo: Random Forest


KeyboardInterrupt: 

In [ ]:
# ==========================================
# 6. Tabla comparativa de resultados
# ==========================================

tabla_resultados = pd.DataFrame(resultados)
tabla_resultados = tabla_resultados.sort_values(by="RMSE")
tabla_resultados.reset_index(drop=True, inplace=True)
tabla_resultados


,Modelo,Mejores parámetros,MAE,RMSE
0,Regresión Lineal,"{'fit_intercept': True, 'positive': True}",1.000656,1.15809


In [ ]:
# ==========================================
# 7. Conclusiones del experimento
# ==========================================

mejor = tabla_resultados.iloc[0]
print(f"El mejor modelo fue: {mejor['Modelo']}")
print(f"Parámetros óptimos: {mejor['Mejores parámetros']}")
print(f"MAE: {mejor['MAE']:.4f}, RMSE: {mejor['RMSE']:.4f}")

print("\nConclusiones:")
print("- La regresión lineal sirve como línea base, pero suele tener mayor error si la relación no es lineal.")
print("- Random Forest ofrece un buen equilibrio entre precisión y generalización.")
print("- XGBoost suele ser el más preciso al manejar relaciones complejas, con costo computacional medio.")
print("- Para mejorar resultados, podría ajustarse el preprocesamiento, eliminar ruido o usar más datos.")


🏆 El mejor modelo fue: Regresión Lineal
Parámetros óptimos: {'fit_intercept': True, 'positive': True}
MAE: 1.0007, RMSE: 1.1581

Conclusiones:
- La regresión lineal sirve como línea base, pero suele tener mayor error si la relación no es lineal.
- Random Forest ofrece un buen equilibrio entre precisión y generalización.
- XGBoost suele ser el más preciso al manejar relaciones complejas, con costo computacional medio.
- Para mejorar resultados, podría ajustarse el preprocesamiento, eliminar ruido o usar más datos.
